In [1]:
import pandas as pd
import re
import os
from datetime import datetime


# 文件夹路径，包含所有要处理的Excel文件
folder_path = "D:\Test\交易平台导出上网电量test\导出于2025年9月22日"

# 定义一个正则表达式，匹配以数字开头并以"日"结尾的工作表名称
pattern = re.compile(r'^\d+日$')

# 创建一个空的DataFrame用于存放所有合并后的数据
combined_df = pd.DataFrame()


In [2]:
# 遍历文件夹中的每一个Excel文件
for file_name in os.listdir(folder_path):
    if file_name.endswith('.xlsx'):
        file_path = os.path.join(folder_path, file_name)

        # 使用ExcelFile读取文件
        xls = pd.ExcelFile(file_path)

        # 遍历Excel文件中的每一个工作表
        for sheet_name in xls.sheet_names:
            # 如果工作表名匹配固定的格式（1日, 2日, ..., 30日）
            if pattern.match(sheet_name):
                # 读取符合条件的工作表
                df = pd.read_excel(xls, sheet_name=sheet_name)
                # 将数据追加到combined_df中
                combined_df = pd.concat([combined_df, df], ignore_index=True)

D:\Develop\Anaconda\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
D:\Develop\Anaconda\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# 假设 df 的时间列名是 “时间”，先确保是 datetime 类型
combined_df['时间'] = pd.to_datetime(combined_df['时间'])
# 将时间列往前挪一个小时对齐时间戳
combined_df['时间'] = combined_df['时间'] - pd.Timedelta(hours=1)
# 新增两列：日期（年月日），时间（时:分:秒）
combined_df['日期'] = combined_df['时间'].dt.date          # 只要年月日
combined_df['小时'] = combined_df['时间'].dt.time        # 只要时分秒
# 假设你的表原来第一列是“现货结算单元”
first_col = combined_df.columns[0]

# 构建新的列顺序：
new_order = (
    [first_col, '日期', '小时'] +
    [c for c in combined_df.columns if c not in [first_col, '日期', '小时']]
)

combined_df = combined_df[new_order]
combined_df = combined_df.drop(columns='时间')



In [4]:
# 获取当前时间作为文件名的一部分
current_time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
# print(current_time)
# # 指定导出文件夹路径
output_folder = folder_path
# # 构造带有当前时间的文件名
output_file = os.path.join(output_folder, f'汇总导出上网电量_{current_time}.xlsx')

# # 将合并后的数据写入到一个新的Excel文件
combined_df.to_excel(output_file, index=False)
print("--------完成合并-------------")


--------完成合并-------------


In [5]:
combined_df

,现货结算单元,日期,小时,上网电量
0,哈密市振超风力发电有限公司（振超分散式兴业风电一场）,2025-09-01,00:00:00,3.287
1,哈密市振超风力发电有限公司（振超分散式兴业风电一场）,2025-09-01,01:00:00,3.738
2,哈密市振超风力发电有限公司（振超分散式兴业风电一场）,2025-09-01,02:00:00,3.350
3,哈密市振超风力发电有限公司（振超分散式兴业风电一场）,2025-09-01,03:00:00,1.260
4,哈密市振超风力发电有限公司（振超分散式兴业风电一场）,2025-09-01,04:00:00,3.581
...,...,...,...,...
811,柯坪县柯特新能源有限责任公司（柯特新能源柯坪光伏一电站）,2025-09-17,19:00:00,13.650
812,柯坪县柯特新能源有限责任公司（柯特新能源柯坪光伏一电站）,2025-09-17,20:00:00,1.722
813,柯坪县柯特新能源有限责任公司（柯特新能源柯坪光伏一电站）,2025-09-17,21:00:00,7.014
814,柯坪县柯特新能源有限责任公司（柯特新能源柯坪光伏一电站）,2025-09-17,22:00:00,6.888
